In [1]:
import numpy as np 
import pandas as pd
from sklearn.model_selection import KFold,cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.decomposition import PCA

In [4]:
df = pd.read_csv('post_feature_selection_v2.csv').drop(columns=['store room','floor_category','balcony'])

In [5]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,flat,sector 36,0.82,3.0,2.0,New Property,850.0,0.0,0.0,Low
1,flat,sector 89,0.95,2.0,2.0,New Property,1226.0,1.0,0.0,Low
2,flat,sohna road,0.32,2.0,2.0,New Property,1000.0,0.0,0.0,Low
3,flat,sector 92,1.60,3.0,4.0,Relatively New,1615.0,1.0,1.0,High
4,flat,sector 102,0.48,2.0,2.0,Relatively New,582.0,0.0,0.0,High


In [6]:
#0-> unfurnished
#1-> semi-furnished
#2-> furnished

In [ ]:
#numerical = bedRoom,bathroom,built_up_area,servant room
#ordinal = furnishing_status,luxury_category
#OHE = property_type, sector, agePossesion

In [ ]:
df['agePossession'].replace({'Relatively New':'new',
                             'Moderately Old':'old',
                            'New Property':'new', 
                            'Old Property':'old',
                            'Under Construction':'under construction'}, inplace=True)

In [10]:
df['agePossession'].value_counts()

agePossession
new                   2331
old                    946
under construction     277
Name: count, dtype: int64

In [11]:
df['property_type'].replace({'flat':0,
                             'house':1}, inplace=True)

C:\Users\Hardik\AppData\Local\Temp\ipykernel_25672\1627148773.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['property_type'].replace({'flat':0,


In [13]:
df['luxury_category'].replace({'Low':0,
                             'Medium':1,
                             'High':2}, inplace=True)

C:\Users\Hardik\AppData\Local\Temp\ipykernel_25672\940299270.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['luxury_category'].replace({'Low':0,
C:\Users\Hardik\AppData\Local\Temp\ipykernel_25672\940299270.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['luxury_category'].replace({'Lo

In [14]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,0,sector 36,0.82,3.0,2.0,new,850.0,0.0,0.0,0
1,0,sector 89,0.95,2.0,2.0,new,1226.0,1.0,0.0,0
2,0,sohna road,0.32,2.0,2.0,new,1000.0,0.0,0.0,0
3,0,sector 92,1.60,3.0,4.0,new,1615.0,1.0,1.0,2
4,0,sector 102,0.48,2.0,2.0,new,582.0,0.0,0.0,2


In [16]:
new_df =pd.get_dummies(df,columns=['sector','agePossession'],drop_first=True)

In [17]:
x=new_df.drop(columns=['price'])
y = new_df['price']

In [ ]:
y_log = np.log1p(y)
#to get noraml distribution of target variable

In [ ]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
#it removes the column name so we will add that

In [20]:
x_scaled=pd.DataFrame(x_scaled, columns=x.columns)

In [21]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), x_scaled, y_log, cv=kfold, scoring='r2')

In [22]:
scores.mean(),scores.std()

(np.float64(0.8512613057405425), np.float64(0.016992929105286228))

In [35]:
from sklearn.linear_model import Ridge
lr = LinearRegression()
ridge = Ridge(alpha=0.0001)
ridge.fit(x_scaled,y_log)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",Non

In [36]:
ridge.coef_

array([ 1.20165033e-01,  5.40015843e-02,  6.51193965e-02,  2.10637554e-01,
        5.09461864e-02,  8.28171743e-03,  6.18569538e-03,  9.81907453e-03,
       -2.27138067e-02, -5.19269573e-03,  5.24021239e-03,  2.69997136e-02,
       -2.90643912e-03,  1.96365747e-03, -1.93923489e-02,  5.72112298e-04,
       -1.29874109e-02,  1.68168747e-02,  2.61494229e-02, -1.49661346e-02,
        8.73367597e-03,  1.62285832e-02,  3.08256534e-02,  3.21707775e-02,
       -1.94577320e-02, -1.22504613e-02,  2.87409954e-02,  3.29133160e-03,
        1.39170137e-02,  6.72157638e-03, -9.41795148e-03,  3.54735949e-02,
        2.19590678e-03,  1.77018721e-02,  5.75602841e-02,  7.38833273e-02,
        6.74154978e-03,  4.18539312e-02, -1.27148213e-02,  6.27609996e-03,
        2.24916348e-02,  2.59567879e-02,  2.49440485e-03, -1.14659854e-02,
        1.14294781e-03,  1.21373726e-02,  2.57693909e-03, -2.07747149e-02,
        7.65343901e-03,  1.67855570e-02,  6.03750346e-02,  2.69569636e-02,
        1.58617170e-02,  

In [37]:
coef_df = pd.DataFrame(ridge.coef_.reshape(1,112),columns=x.columns).stack().reset_index().drop(columns=['level_0']).rename(columns={'level_1':'feature',0:'coef'})

In [27]:
import statsmodels.api as sm
x_with_const = sm.add_constant(x_scaled)
model = sm.OLS(y_log, x_with_const).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  price   R-squared:                       0.865
Model:                            OLS   Adj. R-squared:                  0.860
Method:                 Least Squares   F-statistic:                     196.7
Date:                Sun, 29 Mar 2026   Prob (F-statistic):               0.00
Time:                        15:00:42   Log-Likelihood:                 588.22
No. Observations:                3554   AIC:                            -950.4
Df Residuals:                    3441   BIC:                            -252.6
Df Model:                         112                                         
Covariance Type:            nonrobust                                         
====================================================================================================
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
const                                1.0464      0.003    299.336      0.000       1.040       1.053
property_type                        0.1202      0.005     22.829      0.000       0.110       0.130
bedRoom                              0.0540      0.007      8.196      0.000       0.041       0.067
bathroom                             0.0651      0.007      9.554      0.000       0.052       0.078
built_up_area                        0.2106      0.005     41.763      0.000       0.201       0.221
servant room                         0.0509      0.005     11.231      0.000       0.042       0.060
furnishing_type                      0.0083      0.004      2.038      0.042       0.000       0.016
luxury_category                      0.0062      0.004      1.534      0.125      -0.002       0.014
sector_gwal pahari                   0.0098      0.006      1.551      0.121      -0.003       0.022
sector_manesar                      -0.0227      0.008     -2.922      0.004      -0.038      -0.007
sector_sector 1                     -0.0052      0.005     -1.116      0.264      -0.014       0.004
sector_sector 10                     0.0052      0.005      1.088      0.277      -0.004       0.015
sector_sector 102                    0.0270      0.013      2.049      0.041       0.001       0.053
sector_sector 103                   -0.0029      0.009     -0.332      0.740      -0.020       0.014
sector_sector 104                    0.0020      0.011      0.184      0.854      -0.019       0.023
sector_sector 105                   -0.0194      0.007     -2.926      0.003      -0.032      -0.006
sector_sector 106                    0.0006      0.008      0.068      0.946      -0.016       0.017
sector_sector 107                   -0.0130      0.010     -1.283      0.200      -0.033       0.007
sector_sector 108                    0.0168      0.010      1.674      0.094      -0.003       0.037
sector_sector 109                    0.0261      0.012      2.182      0.029       0.003       0.050
sector_sector 11                    -0.0150      0.006     -2.442      0.015      -0.027      -0.003
sector_sector 110                    0.0087      0.007      1.262      0.207      -0.005       0.022
sector_sector 111                    0.0162      0.007      2.392      0.017       0.003       0.030
sector_sector 112                    0.0308      0.007      4.445      0.000       0.017       0.044
sector_sector 113                    0.0322      0.009      3.644      0.000       0.015       0.049
sector_sector 12                    -0.0195      0.008     -2.469      0.014      -0.035      -0.004
sector_sector 13                    -0.0123      0.005     -2.372      0.018      -0.022      -0.002
sector_sector 14                     0.0287      0.00

As we have done scaling on x and also log transformation on y that's why we can't directly say that the relation btw feature and target is the coeefienct  

In [28]:
y_log.std()

np.float64(0.5579613263072812)

In [29]:
x['bedRoom'].std()

np.float64(1.2455995038118572)

In [31]:
0.054*(0.557/1.245)

0.024159036144578313

In [32]:
np.expm1(0.0241)

np.float64(0.024392752044032903)